# Notebook B - Task 4 (FS2K face-to-sketch conditional GAN)
Run on **Google Colab** (Runtime -> Change runtime type -> **T4 GPU**) or Kaggle.
Put **FS2K.zip** in the root of your Google Drive ("My Drive") first. Run all cells (~30-45 min with `BUDGET=fast`).
At the end **outputs_B.zip** downloads (and is also copied to your Drive).

In [ ]:
# 1) Check GPU
!nvidia-smi
import os, sys
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.isdir('/kaggle/working')
ON_COLAB = (not ON_KAGGLE) and ('google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ)
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
print('kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local', WORK)

In [ ]:
# 2) Get the code + install the few packages Kaggle/Colab do not have
os.chdir(WORK)
!rm -rf GenAI-Assignment1 && git clone -q https://github.com/maima2024/GenAI-Assignment1.git
os.chdir(os.path.join(WORK, 'GenAI-Assignment1'))
print(os.getcwd())
!pip install -q optuna mlflow onnx onnxruntime onnxscript
os.environ['BUDGET'] = 'fast'          # 'smoke' = 2-minute test, 'fast' = assignment run, 'full' = longer
os.environ['OUT_DIR'] = 'outputs'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'

In [ ]:
# 3) FS2K data: from Google Drive (Colab) or an attached Kaggle dataset
os.makedirs('data/raw', exist_ok=True)
import glob, shutil
if ON_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    # look for FS2K*.zip anywhere in My Drive (first 3 folder levels)
    hits = [p for d in range(4) for p in glob.glob('/content/drive/MyDrive/' + '*/' * d + 'FS2K*.zip')]
    print('found in Drive:', hits)
    if hits:
        shutil.copy(hits[0], 'data/raw/FS2K.zip')
    else:
        print('FS2K.zip not found in Drive - please choose it from your computer (upload takes a few minutes)')
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        shutil.move(name, 'data/raw/FS2K.zip')
    print('zip size MB:', os.path.getsize('data/raw/FS2K.zip') / 1e6)
os.environ['FS2K_SEARCH'] = 'data/raw' + (':/kaggle/input' if ON_KAGGLE else '')
!python -c "import sys,os; sys.path.insert(0,'src'); from genai.fs2k_data import build_cache, load; build_cache(os.environ['FS2K_SEARCH'].split(':')); d=load(); print({k:v.shape for k,v in d.items() if k!='split'}, len(d['split']['train']), len(d['split']['val']))"

In [ ]:
# 4) Task 4 - Optuna search, final training, test evaluation, ONNX export
!python scripts/task4_cgan.py

In [ ]:
# 5) Package results
!cd outputs && zip -qr ../outputs_B.zip .
!ls -lh outputs_B.zip outputs/onnx
if ON_COLAB:
    !cp outputs_B.zip /content/drive/MyDrive/outputs_B.zip
    from google.colab import files; files.download('outputs_B.zip')
elif ON_KAGGLE:
    from IPython.display import FileLink; display(FileLink('outputs_B.zip'))